# Build Provincial Masters 2009–2025

Construye los datasets maestros a nivel provincial para **SC Tenerife** (TFE + La Palma + Gomera) y **Las Palmas** (GC + Lanzarote + Fuerteventura) para el período 2009–2025, a partir de los masters insulares existentes.

**Output:**
- `data/processed/provinces/master_provincial_sc_tenerife_2009_2025.parquet`
- `data/processed/provinces/master_provincial_las_palmas_2009_2025.parquet`

**Metodología:**
- Muertes: suma directa de `deaths_week` por isla del grupo
- Temperatura: promedio ponderado por población (`temp_c_mean`)
- Calima proxy: promedio ponderado por población de `calima_proxy_score` insular (proxy v2, AUC 0.886)
- Período: 2009–2025 (887 semanas), consistente con el modelo CCAA (`proxy_reliable == 1`)

In [1]:
import pandas as pd

BASE = r"C:\Users\fdora\RA_Career\Projects\climate_mortality"

# ── CONFIGURACIÓN PROVINCIAL ──────────────────────────────────────────────────
PROVINCES = {
    "sc_tenerife": ["tenerife", "la_palma", "gomera"],
    "las_palmas":  ["gran_canaria", "lanzarote", "fuerteventura"],
}

ISLAND_PATHS = {
    "tenerife":      f"{BASE}/data/processed/tenerife/master/master_tfe_2004_2025.parquet",
    "gran_canaria":  f"{BASE}/data/processed/gran_canaria/master/master_gcan_2004_2025.parquet",
    "la_palma":      f"{BASE}/data/processed/la_palma/master/master_lpa_2004_2025.parquet",
    "gomera":        f"{BASE}/data/processed/gomera/master/master_gom_2004_2025.parquet",
    "lanzarote":     f"{BASE}/data/processed/lanzarote/master/master_lzt_2004_2025.parquet",
    "fuerteventura": f"{BASE}/data/processed/fuerteventura/master/master_ftv_2004_2025.parquet",
}

# ── PASO 1: CARGAR Y FILTRAR 2009+ ────────────────────────────────────────────
island_dfs = {}
for island, path in ISLAND_PATHS.items():
    df = pd.read_parquet(path)
    df["week_start"] = pd.to_datetime(df["week_start"])
    df = df[df["week_start"] >= "2009-01-01"].copy()
    island_dfs[island] = df
    print(f"{island}: {df.shape[0]} semanas | deaths nulls: {df['deaths_week'].isnull().sum()}")

# ── PASO 1: AGREGAR MUERTES POR PROVINCIA ─────────────────────────────────────
deaths_by_province = {}
for prov, islands in PROVINCES.items():
    # Concatenar islas de la provincia y sumar muertes por semana
    combined = pd.concat([island_dfs[i][["week_start", "deaths_week", "deaths_missing_week"]]
                          for i in islands])
    deaths_prov = combined.groupby("week_start", as_index=False).agg(
        deaths        = ("deaths_week",         "sum"),
        deaths_missing = ("deaths_missing_week", "sum"),
    )
    deaths_prov = deaths_prov.sort_values("week_start").reset_index(drop=True)
    deaths_by_province[prov] = deaths_prov
    print(f"\n{prov}: {deaths_prov.shape[0]} semanas | deaths mean={deaths_prov['deaths'].mean():.1f}")
    print(deaths_prov.head(3))

tenerife: 887 semanas | deaths nulls: 0
gran_canaria: 887 semanas | deaths nulls: 0
la_palma: 887 semanas | deaths nulls: 0
gomera: 887 semanas | deaths nulls: 37
lanzarote: 887 semanas | deaths nulls: 0
fuerteventura: 887 semanas | deaths nulls: 4

sc_tenerife: 887 semanas | deaths mean=149.4
  week_start  deaths  deaths_missing
0 2009-01-05   157.0             0.0
1 2009-01-12   147.0             0.0
2 2009-01-19   177.0             0.0

las_palmas: 887 semanas | deaths mean=147.8
  week_start  deaths  deaths_missing
0 2009-01-05   153.0             0.0
1 2009-01-12   155.0             0.0
2 2009-01-19   151.0             0.0


In [2]:
# ── QA: verificar período de cobertura real por isla ─────────────────────────
for island, df in island_dfs.items():
    first_valid = df.loc[df["deaths_week"].notna(), "week_start"].min()
    n_nulls = df["deaths_week"].isnull().sum()
    print(f"{island}: primer dato válido = {first_valid.date()} | nulls = {n_nulls}")

tenerife: primer dato válido = 2009-01-05 | nulls = 0
gran_canaria: primer dato válido = 2009-01-05 | nulls = 0
la_palma: primer dato válido = 2009-01-05 | nulls = 0
gomera: primer dato válido = 2009-01-05 | nulls = 37
lanzarote: primer dato válido = 2009-01-05 | nulls = 0
fuerteventura: primer dato válido = 2009-01-05 | nulls = 4


In [3]:
# ¿Los nulls están concentrados o dispersos?
for island in ["gomera", "fuerteventura"]:
    null_weeks = island_dfs[island].loc[
        island_dfs[island]["deaths_week"].isnull(), "week_start"
    ]
    print(f"\n{island} — {len(null_weeks)} semanas con null:")
    print(f"  Rango: {null_weeks.min().date()} → {null_weeks.max().date()}")
    print(f"  Primeras 5: {null_weeks.dt.date.tolist()[:5]}")


gomera — 37 semanas con null:
  Rango: 2009-02-23 → 2023-07-24
  Primeras 5: [datetime.date(2009, 2, 23), datetime.date(2009, 7, 6), datetime.date(2010, 1, 4), datetime.date(2010, 5, 17), datetime.date(2010, 6, 28)]

fuerteventura — 4 semanas con null:
  Rango: 2009-06-22 → 2011-07-18
  Primeras 5: [datetime.date(2009, 6, 22), datetime.date(2010, 11, 22), datetime.date(2011, 4, 18), datetime.date(2011, 7, 18)]


In [4]:
# confirmado -son 0 reales. 
# ── FIX: rellenar nulls con 0 (semanas sin muertes registradas) ───────────────
for island in island_dfs:
    island_dfs[island]["deaths_week"] = island_dfs[island]["deaths_week"].fillna(0)
    island_dfs[island]["deaths_missing_week"] = island_dfs[island]["deaths_missing_week"].fillna(0)

# ── REHACER AGREGACIÓN PROVINCIAL ────────────────────────────────────────────
deaths_by_province = {}
for prov, islands in PROVINCES.items():
    combined = pd.concat([island_dfs[i][["week_start", "deaths_week", "deaths_missing_week"]]
                          for i in islands])
    deaths_prov = combined.groupby("week_start", as_index=False).agg(
        deaths         = ("deaths_week",         "sum"),
        deaths_missing = ("deaths_missing_week", "sum"),
    )
    deaths_prov = deaths_prov.sort_values("week_start").reset_index(drop=True)
    deaths_by_province[prov] = deaths_prov
    print(f"{prov}: {deaths_prov.shape[0]} semanas | deaths mean={deaths_prov['deaths'].mean():.1f} | nulls={deaths_prov['deaths'].isnull().sum()}")

sc_tenerife: 887 semanas | deaths mean=149.4 | nulls=0
las_palmas: 887 semanas | deaths mean=147.8 | nulls=0


In [5]:
# ── PASO 2: TEMPERATURA PONDERADA POR POBLACIÓN ───────────────────────────────

# Pobaciones 2009–2025 (usamos población media por isla como peso fijo)
# Fuente: INE / padrón municipal
POP = {
    "tenerife":      917_841,
    "gran_canaria":  851_231,
    "la_palma":       82_671,
    "gomera":         21_503,
    "lanzarote":     155_812,
    "fuerteventura": 119_732,
}

def weighted_temp(province_islands):
    frames = []
    total_pop = sum(POP[i] for i in province_islands)
    for island in province_islands:
        df = island_dfs[island][["week_start", "temp_c_mean"]].copy()
        df["temp_weighted"] = df["temp_c_mean"] * (POP[island] / total_pop)
        frames.append(df[["week_start", "temp_weighted"]])
    combined = pd.concat(frames)
    return combined.groupby("week_start", as_index=False).agg(
        temp_c_mean=("temp_weighted", "sum")
    ).sort_values("week_start").reset_index(drop=True)

temp_by_province = {}
for prov, islands in PROVINCES.items():
    temp_by_province[prov] = weighted_temp(islands)
    nulls = temp_by_province[prov]["temp_c_mean"].isnull().sum()
    mean  = temp_by_province[prov]["temp_c_mean"].mean()
    print(f"{prov}: temp mean={mean:.2f}°C | nulls={nulls}")

sc_tenerife: temp mean=21.78°C | nulls=0
las_palmas: temp mean=21.76°C | nulls=0


In [6]:
# ── PASO 3: CALIMA PROXY PONDERADO POR POBLACIÓN ─────────────────────────────

CALIMA_PATHS = {
    "tenerife":      f"{BASE}/data/processed/tenerife/calima/calima_proxy_weekly_tfe_2004_2025.parquet",
    "gran_canaria":  f"{BASE}/data/processed/gran_canaria/calima/calima_proxy_weekly_gcan_2004_2025.parquet",
    "la_palma":      f"{BASE}/data/processed/la_palma/calima/calima_proxy_weekly_lpa_2004_2025.parquet",
    "gomera":        f"{BASE}/data/processed/gomera/calima/calima_proxy_weekly_gom_2004_2025.parquet",
    "lanzarote":     f"{BASE}/data/processed/lanzarote/calima/calima_proxy_weekly_lzt_2004_2025.parquet",
    "fuerteventura": f"{BASE}/data/processed/fuerteventura/calima/calima_proxy_weekly_ftv_2004_2025.parquet",
}

# Cargar proxies insulares filtrados a 2009+
calima_dfs = {}
for island, path in CALIMA_PATHS.items():
    df = pd.read_parquet(path)
    df["week_start"] = pd.to_datetime(df["week_start"])
    calima_dfs[island] = df[df["week_start"] >= "2009-01-01"].copy()

def weighted_calima(province_islands):
    total_pop = sum(POP[i] for i in province_islands)
    frames = []
    for island in province_islands:
        df = calima_dfs[island][["week_start", "calima_proxy_score"]].copy()
        df["score_weighted"] = df["calima_proxy_score"] * (POP[island] / total_pop)
        frames.append(df[["week_start", "score_weighted"]])
    combined = pd.concat(frames)
    result = combined.groupby("week_start", as_index=False).agg(
        calima_score_provincial=("score_weighted", "sum")
    ).sort_values("week_start").reset_index(drop=True)

    # Derivar nivel categórico (mismo criterio que proxy v2)
    bins  = [-0.001, 0.25, 0.50, 0.75, 1.001]
    labels = ["no_calima", "possible", "probable", "intense"]
    result["calima_level_provincial"] = pd.cut(
        result["calima_score_provincial"], bins=bins, labels=labels
    )
    return result

calima_by_province = {}
for prov, islands in PROVINCES.items():
    calima_by_province[prov] = weighted_calima(islands)
    dist = calima_by_province[prov]["calima_level_provincial"].value_counts().to_dict()
    print(f"{prov}: score mean={calima_by_province[prov]['calima_score_provincial'].mean():.3f} | dist={dist}")

sc_tenerife: score mean=0.214 | dist={'no_calima': 619, 'possible': 132, 'probable': 77, 'intense': 59}
las_palmas: score mean=0.250 | dist={'no_calima': 574, 'possible': 175, 'probable': 86, 'intense': 52}


In [7]:
# ── PASO 4: UNIR Y GUARDAR ────────────────────────────────────────────────────
import os
os.makedirs(f"{BASE}/data/processed/provinces", exist_ok=True)

for prov in PROVINCES:
    master = (
        deaths_by_province[prov]
        .merge(temp_by_province[prov],   on="week_start", how="left")
        .merge(calima_by_province[prov], on="week_start", how="left")
    )
    master["province"] = prov

    # Verificación rápida
    assert master.shape[0] == 887, f"Error: {prov} tiene {master.shape[0]} filas, esperadas 887"
    assert master.isnull().sum().sum() == 0, f"Error: nulls en {prov}:\n{master.isnull().sum()}"

    out = f"{BASE}/data/processed/provinces/master_provincial_{prov}_2009_2025.parquet"
    master.to_parquet(out, index=False)
    print(f"✅ {prov}: {master.shape} guardado → {out}")
    print(f"   Columnas: {master.columns.tolist()}")
    print(f"   Rango: {master.week_start.min().date()} → {master.week_start.max().date()}")
    print()

✅ sc_tenerife: (887, 7) guardado → C:\Users\fdora\RA_Career\Projects\climate_mortality/data/processed/provinces/master_provincial_sc_tenerife_2009_2025.parquet
   Columnas: ['week_start', 'deaths', 'deaths_missing', 'temp_c_mean', 'calima_score_provincial', 'calima_level_provincial', 'province']
   Rango: 2009-01-05 → 2025-12-29

✅ las_palmas: (887, 7) guardado → C:\Users\fdora\RA_Career\Projects\climate_mortality/data/processed/provinces/master_provincial_las_palmas_2009_2025.parquet
   Columnas: ['week_start', 'deaths', 'deaths_missing', 'temp_c_mean', 'calima_score_provincial', 'calima_level_provincial', 'province']
   Rango: 2009-01-05 → 2025-12-29



## Output generado

| Archivo | Filas | Período | Columnas |
|---|---|---|---|
| `master_provincial_sc_tenerife_2009_2025.parquet` | 887 | 2009-01-05 → 2025-12-29 | 7 |
| `master_provincial_las_palmas_2009_2025.parquet` | 887 | 2009-01-05 → 2025-12-29 | 7 |

**Notas:**
- Muertes: suma directa por isla. Gomera (37 nulls) y Fuerteventura (4 nulls) rellenados con 0 — semanas sin muertes en islas de baja población.
- Temperatura: promedio ponderado por población (pesos fijos INE).
- Calima proxy: promedio ponderado por población de `calima_proxy_score` insular (proxy v2, AUC 0.886). Nivel categórico derivado con los mismos bins que el proxy regional.
- Gomera representa ~1% del peso provincial de SC Tenerife — impacto de sus nulls despreciable.